# FreshLens — DishRecog Model Training
**MobileNetV3 fine-tuned on Food-101 → exported to TFLite INT8**
Runtime: ~45 mins on T4 GPU (free Colab)

## Steps
1. Run all cells top to bottom
2. Download `dishrecog.tflite` from the output
3. Push to `ml_models/` in your GitHub repo

In [ ]:
# Cell 1: Install dependencies
!pip install tensorflow==2.13.0 tensorflow-datasets matplotlib seaborn -q

In [ ]:
# Cell 2: Imports
import tensorflow as tf
import tensorflow_datasets as tfds
import numpy as np
import matplotlib.pyplot as plt
import os

print(f'TensorFlow: {tf.__version__}')
print(f'GPU: {tf.config.list_physical_devices("GPU")}')

In [ ]:
# Cell 3: Config
IMG_SIZE = 224
BATCH_SIZE = 64
EPOCHS_FREEZE = 5      # Train only head
EPOCHS_UNFREEZE = 10   # Fine-tune top layers
NUM_CLASSES = 101
AUTOTUNE = tf.data.AUTOTUNE
print('Config set')

In [ ]:
# Cell 4: Load Food-101 dataset
(ds_train, ds_val, ds_test), info = tfds.load(
    'food101',
    split=['train[:80%]', 'train[80%:]', 'validation'],
    as_supervised=True,
    with_info=True
)

CLASS_NAMES = info.features['label'].names
print(f'Classes: {len(CLASS_NAMES)}')
print(f'Train size: {info.splits["train"].num_examples}')

In [ ]:
# Cell 5: Preprocessing
def preprocess(image, label):
    image = tf.image.resize(image, [IMG_SIZE, IMG_SIZE])
    image = tf.cast(image, tf.float32) / 255.0
    return image, label

def augment(image, label):
    image = tf.image.random_flip_left_right(image)
    image = tf.image.random_brightness(image, 0.2)
    image = tf.image.random_contrast(image, 0.8, 1.2)
    image = tf.image.random_saturation(image, 0.8, 1.2)
    return image, label

train_ds = ds_train.map(preprocess, num_parallel_calls=AUTOTUNE)\
                   .map(augment, num_parallel_calls=AUTOTUNE)\
                   .shuffle(1000).batch(BATCH_SIZE).prefetch(AUTOTUNE)

val_ds = ds_val.map(preprocess, num_parallel_calls=AUTOTUNE)\
               .batch(BATCH_SIZE).prefetch(AUTOTUNE)

test_ds = ds_test.map(preprocess, num_parallel_calls=AUTOTUNE)\
                 .batch(BATCH_SIZE).prefetch(AUTOTUNE)

print('Datasets ready')

In [ ]:
# Cell 6: Build MobileNetV3 model
base_model = tf.keras.applications.MobileNetV3Large(
    input_shape=(IMG_SIZE, IMG_SIZE, 3),
    include_top=False,
    weights='imagenet',
    include_preprocessing=False
)
base_model.trainable = False  # Freeze for initial training

inputs = tf.keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
x = base_model(inputs, training=False)
x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.Dropout(0.3)(x)
x = tf.keras.layers.Dense(512, activation='relu')(x)
x = tf.keras.layers.Dropout(0.2)(x)
outputs = tf.keras.layers.Dense(NUM_CLASSES, activation='softmax')(x)

model = tf.keras.Model(inputs, outputs)
model.summary()

In [ ]:
# Cell 7: Phase 1 — Train head only
model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

callbacks = [
    tf.keras.callbacks.EarlyStopping(patience=3, restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(factor=0.5, patience=2)
]

history1 = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS_FREEZE, callbacks=callbacks)
print('Phase 1 done')

In [ ]:
# Cell 8: Phase 2 — Fine-tune top 30 layers
base_model.trainable = True
for layer in base_model.layers[:-30]:
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-5),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

history2 = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS_UNFREEZE, callbacks=callbacks)
print('Phase 2 done')

In [ ]:
# Cell 9: Evaluate
loss, acc = model.evaluate(test_ds)
print(f'Test Accuracy: {acc*100:.2f}%')
print(f'Test Loss: {loss:.4f}')

In [ ]:
# Cell 10: Save class names for Android
import json
with open('food101_labels.txt', 'w') as f:
    f.write('\n'.join(CLASS_NAMES))
with open('food101_labels.json', 'w') as f:
    json.dump(CLASS_NAMES, f)
print('Labels saved')

In [ ]:
# Cell 11: Export to TFLite with INT8 quantization
def representative_dataset():
    for images, _ in train_ds.take(100):
        for img in images:
            yield [tf.expand_dims(img, 0)]

# Full INT8 quantization (4x smaller, faster on mobile)
converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.representative_dataset = representative_dataset
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter.inference_input_type = tf.uint8
converter.inference_output_type = tf.uint8

tflite_model = converter.convert()

with open('dishrecog.tflite', 'wb') as f:
    f.write(tflite_model)

size_mb = len(tflite_model) / (1024 * 1024)
print(f'TFLite model saved: {size_mb:.2f} MB')

In [ ]:
# Cell 12: Verify TFLite model works
interpreter = tf.lite.Interpreter(model_content=tflite_model)
interpreter.allocate_tensors()

input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

print('Input:', input_details[0]['shape'], input_details[0]['dtype'])
print('Output:', output_details[0]['shape'], output_details[0]['dtype'])

# Test with random input
test_input = np.random.randint(0, 255, (1, IMG_SIZE, IMG_SIZE, 3), dtype=np.uint8)
interpreter.set_tensor(input_details[0]['index'], test_input)
interpreter.invoke()
output = interpreter.get_tensor(output_details[0]['index'])
predicted_class = CLASS_NAMES[np.argmax(output)]
print(f'Test inference OK — predicted: {predicted_class}')

In [ ]:
# Cell 13: Download files
from google.colab import files
files.download('dishrecog.tflite')   # Main model — put in ml_models/
files.download('food101_labels.txt') # Labels — put in app/src/main/assets/
files.download('food101_labels.json')
print('Download started!')

## After Download
1. Copy `dishrecog.tflite` → `ml_models/dishrecog.tflite`
2. Copy `food101_labels.txt` → `app/src/main/assets/food101_labels.txt`
3. `git add . && git commit -m 'feat: add trained DishRecog TFLite model (Food-101, MobileNetV3)' && git push`